# Gráfico de anomalías — RSNA Knee Abnormality Detection

El trabajo original ya contenía filtrado de etiquetas completas y un gráfico.
Sus salidas guardadas informaban 58 estudios completos y 8 con una sola etiqueta positiva.
Esas cifras son **históricas**: no se han vuelto a verificar sin el archivo original.

Corrección del 27/10/2026: se lee únicamente `train.csv`, sin descargar todas las
imágenes de la competencia. Descargar el CSV con la cuenta propia de Kaggle,
aceptando sus condiciones, y guardarlo en `datos/crudos/rsna/train.csv`.
No subir el CSV, imágenes ni identificadores de estudios a este repositorio.
Se puede elegir otra ruta en la variable `ruta`.


In [ ]:
from pathlib import Path
import sys
# Funciona desde la raíz y desde las subcarpetas del repositorio.
candidatos = [Path.cwd(), *Path.cwd().parents]
RAIZ = next((p for p in candidatos if (p / "funciones/datos.py").is_file()), None)
if RAIZ is None:
    raise FileNotFoundError("Abrir el notebook dentro de la carpeta Class-activities.")
sys.path.insert(0, str(RAIZ))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
ruta = RAIZ / "datos/crudos/rsna/train.csv"
LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
          "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
train = None
completas = None
if ruta.is_file():
    train = pd.read_csv(ruta)
    faltan = set(LABELS) - set(train.columns)
    if faltan:
        raise ValueError(f"Faltan columnas de etiquetas: {sorted(faltan)}")
    etiquetas = train[LABELS].apply(pd.to_numeric, errors="raise")
    if not etiquetas.isin([0, 1]).where(etiquetas.notna(), True).all().all():
        raise ValueError("Hay etiquetas diferentes de 0, 1 o faltante")
    completas = etiquetas.dropna(subset=LABELS).copy()
    completas["cantidad_anomalias"] = completas[LABELS].sum(axis=1)
    una_anomalia = completas[completas["cantidad_anomalias"] == 1].copy()
    una_anomalia["anomalia"] = una_anomalia[LABELS].idxmax(axis=1)
    print("Estudios con las 12 etiquetas completas:", len(completas))
    print("Estudios con exactamente una etiqueta positiva:", len(una_anomalia))
    print("Faltantes por etiqueta:")
    print(etiquetas.isna().sum())
    print("Recuento agregado de anomalías únicas:")
    print(una_anomalia["anomalia"].value_counts())
else:
    print("Pendiente de reproducir: falta train.csv. Ver instrucciones arriba.")


In [ ]:
if completas is not None:
    conteo_anomalias = completas[LABELS].sum().sort_values(ascending=False)
    assert conteo_anomalias.between(0, len(completas)).all()
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar(conteo_anomalias.index, conteo_anomalias.values)
    ax.set(title=f"Etiquetas positivas en {len(completas)} estudios con etiquetas completas",
           xlabel="Anomalía", ylabel="Cantidad de estudios")
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    fig.text(.01, .01, "Fuente: RSNA/Kaggle. Un estudio puede aportar a varias barras. Muestra de casos completos.", fontsize=9)
    fig.tight_layout(rect=(0, .05, 1, 1))
    plt.show()
else:
    print("Gráfico pendiente: no se simulan datos ni resultados para reemplazar el archivo.")


## Interpretación y límites

Cada barra cuenta estudios con esa etiqueta, dentro de los que tienen las doce completas.
Las barras no son categorías excluyentes: un estudio puede tener varias etiquetas positivas.
Seleccionar casos completos puede introducir sesgo. El resultado no estima por sí solo
prevalencias poblacionales. Las etiquetas y su definición provienen de la competencia;
no se tratan como diagnósticos nuevos validados por este ejercicio.
